In [1]:
import pandas as pd 
import numpy as np

In [2]:
df_board = pd.read_csv('board_members.csv')
df_vendors = pd.read_csv('vendors_intake.csv')
df_hoas = pd.read_csv('hoas_export.csv')

## Board

In [3]:
df_board.head(5)

,hoa_code,full_name,role,email,term_start,term_end
0,SR-01,Alex Tanaka,President,alex.tanaka@example.com,2023-07-01,2025-07-01
1,SR-01,Reese Patel,Vice President,reese.patel@example.com,2024-01-01,NaN
2,SR-01,Alex Nguyen,Treasurer,NaN,2023-07-01,2026-01-01
3,SR-02,Reese Webb,President,reese.webb@example.com,2024-06-15,2025-07-01
4,SR-02,Jamie Flores,Vice President,jamie.flores@example.com,2024-01-01,2025-07-01


In [4]:
df_board.info()

<class 'pandas.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   hoa_code    53 non-null     str  
 1   full_name   53 non-null     str  
 2   role        53 non-null     str  
 3   email       45 non-null     str  
 4   term_start  53 non-null     str  
 5   term_end    34 non-null     str  
dtypes: str(6)
memory usage: 2.6 KB


In [5]:
# Dropping duplicate rows
df_board = df_board.drop_duplicates(keep = 'last')

df_board[df_board.duplicated(keep=False)]

,hoa_code,full_name,role,email,term_start,term_end


In [6]:
# Changing datatypes for term_start and term_end
df_board['term_start'] = pd.to_datetime(df_board['term_start'],
                                       format='mixed',
                                       utc=False)
df_board['term_end'] = pd.to_datetime(df_board['term_end'],
                                       format='mixed',
                                       utc=False)
df_board.dtypes

hoa_code                 str
full_name                str
role                     str
email                    str
term_start    datetime64[us]
term_end      datetime64[us]
dtype: object

In [7]:
df_board["term_start"].apply(type).value_counts()

term_start
<class 'pandas.Timestamp'>    52
Name: count, dtype: int64

In [8]:
df_board["email"].apply(type).value_counts()

email
<class 'str'>      44
<class 'float'>     8
Name: count, dtype: int64

In [9]:
# Replace NaN's as Nonetypes
df_board['email'] = df_board['email'].replace({np.nan: None})
df_board['term_end'] = (
        df_board['term_end']
        .dt.strftime("%Y-%m-%d")                  
        .astype(object)
        .where(df_board['term_end'].notna(), None)       
    )

In [10]:
df_board["email"].apply(type).value_counts()

email
<class 'str'>         44
<class 'NoneType'>     8
Name: count, dtype: int64

In [11]:
df_board.head()

,hoa_code,full_name,role,email,term_start,term_end
1,SR-01,Reese Patel,Vice President,reese.patel@example.com,2024-01-01,None
2,SR-01,Alex Nguyen,Treasurer,None,2023-07-01,2026-01-01
3,SR-02,Reese Webb,President,reese.webb@example.com,2024-06-15,2025-07-01
4,SR-02,Jamie Flores,Vice President,jamie.flores@example.com,2024-01-01,2025-07-01
5,SR-02,Reese Brooks,Treasurer,reese.brooks@example.com,2023-07-01,2025-07-01


In [12]:
df_board.dtypes


hoa_code                 str
full_name                str
role                     str
email                 object
term_start    datetime64[us]
term_end              object
dtype: object

## Vendors

In [13]:
df_vendors.head()

,vendor_name,trade,phone,email,serves_hoa_codes,coi_on_file,service_area
0,Bright Path Landscaping,Landscaping,(555) 201-3344,contact@brightpathscapes.example,SR-01; SR-04; SR-10,Yes,"Bay Harbor, Riverton"
1,Bright Path Landscaping LLC,Landscaping,555-201-3344,contact@brightpathscapes.example,SR-07,Y,Bay Harbor
2,Coastline Plumbing,Plumbing,555.880.1290,dispatch@coastlineplumb.example,SR-04; SR-07,Yes,Bay Harbor
3,Coastline Plumbing,Plumbing,5558801290,dispatch@coastlineplumb.example,SR-10,yes,Bay Harbor
4,Evergreen Tree Care,Tree Service,(555) 442-9981,hello@evergreentree.example,NaN,No,Greenvale


In [14]:
df_vendors.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   vendor_name       14 non-null     str  
 1   trade             14 non-null     str  
 2   phone             14 non-null     str  
 3   email             14 non-null     str  
 4   serves_hoa_codes  9 non-null      str  
 5   coi_on_file       13 non-null     str  
 6   service_area      12 non-null     str  
dtypes: str(7)
memory usage: 916.0 bytes


In [15]:
# Formatting vendor names so that they are consistent based on the email column
df_vendors['vendor_name'] = df_vendors.groupby('email')['vendor_name'].transform('first')
df_vendors.head(2)

,vendor_name,trade,phone,email,serves_hoa_codes,coi_on_file,service_area
0,Bright Path Landscaping,Landscaping,(555) 201-3344,contact@brightpathscapes.example,SR-01; SR-04; SR-10,Yes,"Bay Harbor, Riverton"
1,Bright Path Landscaping,Landscaping,555-201-3344,contact@brightpathscapes.example,SR-07,Y,Bay Harbor


In [16]:
# Formatting phone numbers to make them consistent
df_vendors['phone'] = df_vendors['phone'].str.replace(r'\D', '', regex=True)

df_vendors.head(3)

,vendor_name,trade,phone,email,serves_hoa_codes,coi_on_file,service_area
0,Bright Path Landscaping,Landscaping,5552013344,contact@brightpathscapes.example,SR-01; SR-04; SR-10,Yes,"Bay Harbor, Riverton"
1,Bright Path Landscaping,Landscaping,5552013344,contact@brightpathscapes.example,SR-07,Y,Bay Harbor
2,Coastline Plumbing,Plumbing,5558801290,dispatch@coastlineplumb.example,SR-04; SR-07,Yes,Bay Harbor


In [17]:
# Format coi_on_file column to be boolean 
df_vendors['coi_on_file'] = df_vendors['coi_on_file'].str.lower().str.contains('y', na=False)

In [18]:
# Replace all NaN's with Nonetype
df_vendors['serves_hoa_codes'] = df_vendors['serves_hoa_codes'].replace({np.nan: None})
df_vendors['coi_on_file'] = df_vendors['coi_on_file'].replace({np.nan: None})
df_vendors['service_area'] = df_vendors['service_area'].replace({np.nan: None})

In [19]:
# Convert service_area column into a list instead of continuous string
df_vendors['service_area'] = (
    df_vendors['service_area']
    .fillna('')
    .str.split(',')
    .apply(lambda xs: [x.strip() for x in xs if x.strip()])
)

# Convert serves_hoa_codes into a list instead of a continuous string
df_vendors['serves_hoa_codes'] = (
    df_vendors['serves_hoa_codes']
    .fillna('')
    .str.split(';')
    .apply(lambda xs: [x.strip() for x in xs if x.strip()])
)

In [20]:
df_vendors.dtypes

vendor_name            str
trade                  str
phone                  str
email                  str
serves_hoa_codes    object
coi_on_file           bool
service_area        object
dtype: object

In [21]:
# Create join table that removes duplicates and groups by email to put all hoa codes in one list per vendor
df_vendors_join = df_vendors.groupby('email', as_index=False)['serves_hoa_codes'].agg(lambda x: sorted(set(sum(x, []))))

# Join df_vendors onto df_vendors_join on email and drop duplicates 
df_vendors = (
    df_vendors.drop(columns='serves_hoa_codes')
              .drop_duplicates(subset='email')
              .merge(df_vendors_join, on='email', how='left')
)

df_vendors

,vendor_name,trade,phone,email,coi_on_file,service_area,serves_hoa_codes
0,Bright Path Landscaping,Landscaping,5552013344,contact@brightpathscapes.example,True,"[Bay Harbor, Riverton]","[SR-01, SR-04, SR-07, SR-10]"
1,Coastline Plumbing,Plumbing,5558801290,dispatch@coastlineplumb.example,True,[Bay Harbor],"[SR-04, SR-07, SR-10]"
2,Evergreen Tree Care,Tree Service,5554429981,hello@evergreentree.example,False,[Greenvale],[]
3,Summit Roofing Co.,Roofing,5557770042,jobs@summitroof.example,False,[],[]
4,AquaPro Pool Service,Pool Maintenance,5553321188,service@aquapro.example,True,"[Lakeside, Bay Harbor]","[SR-10, SR-12]"
5,Reliable Electric,Electrical,5559094521,office@reliableelec.example,True,[Riverton],[SR-01]
6,GreenLeaf Pest Control,Pest Control,5556107733,book@greenleafpest.example,True,"[Greenvale, Lakeside]",[]
7,Titan Paving,Asphalt/Paving,5552456610,estimates@titanpaving.example,False,[Bay Harbor],[SR-04]
8,ClearView Window Cleaning,Window Cleaning,5551182200,info@clearviewwc.example,True,[],[]
9,Anchor Security Patrol,Security,5557009090,dispatch@anchorsecurity.example,True,[Bay Harbor],"[SR-04, SR-10]"


## hoas_export

In [22]:
df_hoas

,hoa_code,association_name,city,state,unit_count,monthly_dues,fiscal_year_end,reserve_balance,last_reserve_study,has_reserve_study,board_email
0,SR-01,Maplewood Terrace Owners Association,Riverton,CA,48.0,$285.00,December,"$142,500",2023-04-12,Yes,board@maplewoodterrace.example
1,SR-02,"Cedar Hollow Condominiums, Inc.",Lakeside,Calif.,NaN,310,12,98750,04/2022,yes,cedarhollowhoa@example.com
2,SR-03,Sunfield Villas HOA,Riverton,California,72.0,$340,December,NaN,NaN,No,NaN
3,SR-04,The Beacon Pointe Association,Bay Harbor,CA,120.0,$415.00,June,"$512,000",March 2024,Yes,info@beaconpointe.example
4,SR-05,Orchard Glen Homeowners Assn,Lakeside,CA,36.0,255,12,"61,200",2021-09-01,Yes,board@orchardglen.example
5,SR-06,Stonebridge Commons,Bay Harbor,ca,NaN,$0,December,TBD,NaN,No,stonebridge@example.com
6,SR-07,"Harborview Heights Owners Association, Inc.",Bay Harbor,CA,94.0,$390.00,December,"$378,900",2024-01-15,Yes,hello@harborviewheights.example
7,SR-08,Willow Creek Estates,Greenvale,CA,28.0,220,March,"$44,000",2020-06-30,Yes,NaN
8,SR-09,Pinecrest Gardens Condo Assoc.,Greenvale,CA,55.0,$300,December,NaN,NaN,No,pinecrest@example.com
9,SR-10,Lantern Bay Community Association,Bay Harbor,CA,140.0,$460.00,September,"$690,250",2023-11-20,Yes,board@lanternbay.example


In [23]:
# Make state column consistent 
df_hoas.loc[df_hoas['state'].str.lower().str.contains('ca', na=False), 'state'] = 'CA'

df_hoas.head(2)

,hoa_code,association_name,city,state,unit_count,monthly_dues,fiscal_year_end,reserve_balance,last_reserve_study,has_reserve_study,board_email
0,SR-01,Maplewood Terrace Owners Association,Riverton,CA,48.0,$285.00,December,"$142,500",2023-04-12,Yes,board@maplewoodterrace.example
1,SR-02,"Cedar Hollow Condominiums, Inc.",Lakeside,CA,NaN,310,12,98750,04/2022,yes,cedarhollowhoa@example.com


In [24]:
# Convert unit_count to integers
df_hoas['unit_count'] = df_hoas['unit_count'].astype('Int64')
df_hoas['unit_count'].dtype

Int64Dtype()

In [25]:
# Clean monthly dues and convert to integer
df_hoas['monthly_dues'] = df_hoas['monthly_dues'].replace(0, pd.NA)
df_hoas['monthly_dues'] = df_hoas['monthly_dues'].str.replace('$', '', regex=False)
df_hoas['monthly_dues'] = df_hoas['monthly_dues'].str.replace(',', '', regex=False).apply(float).apply(int)
df_hoas['monthly_dues'] = df_hoas['monthly_dues'].replace(0, pd.NA)

In [26]:
# Clean reserve_balance column and convert to integer
df_hoas['reserve_balance'] = df_hoas['reserve_balance'].str.replace('$', '', regex=False)
df_hoas['reserve_balance'] = df_hoas['reserve_balance'].str.replace(',', '', regex=False)
df_hoas["reserve_balance"] = pd.to_numeric(df_hoas["reserve_balance"], errors="coerce").astype('Int64')


In [27]:
# Convert messy months into consistent format
from datetime import datetime
df_hoas['fiscal_year_end'] = df_hoas['fiscal_year_end'].apply(
    lambda x: 
    pd.NA if pd.isna(x)
    else int(x) if str(x).strip().isdigit()
    else datetime.strptime(str(x).strip(), "%B").month
)
df_hoas.head()

,hoa_code,association_name,city,state,unit_count,monthly_dues,fiscal_year_end,reserve_balance,last_reserve_study,has_reserve_study,board_email
0,SR-01,Maplewood Terrace Owners Association,Riverton,CA,48,285,12,142500,2023-04-12,Yes,board@maplewoodterrace.example
1,SR-02,"Cedar Hollow Condominiums, Inc.",Lakeside,CA,<NA>,310,12,98750,04/2022,yes,cedarhollowhoa@example.com
2,SR-03,Sunfield Villas HOA,Riverton,CA,72,340,12,<NA>,NaN,No,NaN
3,SR-04,The Beacon Pointe Association,Bay Harbor,CA,120,415,6,512000,March 2024,Yes,info@beaconpointe.example
4,SR-05,Orchard Glen Homeowners Assn,Lakeside,CA,36,255,12,61200,2021-09-01,Yes,board@orchardglen.example


In [28]:
# Convert messy dates into consistent format and remove nulls
df_hoas['last_reserve_study'] = pd.to_datetime(df_hoas['last_reserve_study'], format='mixed', utc = False)
df_hoas['last_reserve_study'] = (
        df_hoas['last_reserve_study']
        .dt.strftime("%Y-%m-%d")                  
        .astype(object)
        .where(df_hoas['last_reserve_study'].notna(), None)       
    )

In [29]:
# Convert has_reserve_study into boolean 
df_hoas['has_reserve_study'] = df_hoas['has_reserve_study'].str.lower().str.contains('y', na=False)

In [30]:
# Remove nulls from board_email 
df_hoas['board_email'] = df_hoas['board_email'].replace({np.nan: None})
df_hoas.head()

,hoa_code,association_name,city,state,unit_count,monthly_dues,fiscal_year_end,reserve_balance,last_reserve_study,has_reserve_study,board_email
0,SR-01,Maplewood Terrace Owners Association,Riverton,CA,48,285,12,142500,2023-04-12,True,board@maplewoodterrace.example
1,SR-02,"Cedar Hollow Condominiums, Inc.",Lakeside,CA,<NA>,310,12,98750,2022-04-01,True,cedarhollowhoa@example.com
2,SR-03,Sunfield Villas HOA,Riverton,CA,72,340,12,<NA>,None,False,None
3,SR-04,The Beacon Pointe Association,Bay Harbor,CA,120,415,6,512000,2024-03-01,True,info@beaconpointe.example
4,SR-05,Orchard Glen Homeowners Assn,Lakeside,CA,36,255,12,61200,2021-09-01,True,board@orchardglen.example


## Create Junction table

In [31]:
df_vendors

,vendor_name,trade,phone,email,coi_on_file,service_area,serves_hoa_codes
0,Bright Path Landscaping,Landscaping,5552013344,contact@brightpathscapes.example,True,"[Bay Harbor, Riverton]","[SR-01, SR-04, SR-07, SR-10]"
1,Coastline Plumbing,Plumbing,5558801290,dispatch@coastlineplumb.example,True,[Bay Harbor],"[SR-04, SR-07, SR-10]"
2,Evergreen Tree Care,Tree Service,5554429981,hello@evergreentree.example,False,[Greenvale],[]
3,Summit Roofing Co.,Roofing,5557770042,jobs@summitroof.example,False,[],[]
4,AquaPro Pool Service,Pool Maintenance,5553321188,service@aquapro.example,True,"[Lakeside, Bay Harbor]","[SR-10, SR-12]"
5,Reliable Electric,Electrical,5559094521,office@reliableelec.example,True,[Riverton],[SR-01]
6,GreenLeaf Pest Control,Pest Control,5556107733,book@greenleafpest.example,True,"[Greenvale, Lakeside]",[]
7,Titan Paving,Asphalt/Paving,5552456610,estimates@titanpaving.example,False,[Bay Harbor],[SR-04]
8,ClearView Window Cleaning,Window Cleaning,5551182200,info@clearviewwc.example,True,[],[]
9,Anchor Security Patrol,Security,5557009090,dispatch@anchorsecurity.example,True,[Bay Harbor],"[SR-04, SR-10]"


In [32]:
# Junction table that uses email and hoa_code to connect vendors_intake to hoas_export
df_junction = df_vendors[['email', 'serves_hoa_codes']].explode('serves_hoa_codes', ignore_index = True).dropna()
df_junction = df_junction.rename(columns={'serves_hoa_codes': 'hoa_code'})

# Drop serves_hoa_codes from vendors dataframe
df_vendors = df_vendors.drop(columns = 'serves_hoa_codes')

df_vendors.head(1)

,vendor_name,trade,phone,email,coi_on_file,service_area
0,Bright Path Landscaping,Landscaping,5552013344,contact@brightpathscapes.example,True,"[Bay Harbor, Riverton]"


In [33]:
df_junction

,email,hoa_code
0,contact@brightpathscapes.example,SR-01
1,contact@brightpathscapes.example,SR-04
2,contact@brightpathscapes.example,SR-07
3,contact@brightpathscapes.example,SR-10
4,dispatch@coastlineplumb.example,SR-04
5,dispatch@coastlineplumb.example,SR-07
6,dispatch@coastlineplumb.example,SR-10
9,service@aquapro.example,SR-10
10,service@aquapro.example,SR-12
11,office@reliableelec.example,SR-01


## Pushing data to Supabase

In [34]:
import os
from dotenv import load_dotenv
from supabase import create_client

In [35]:
# Load in environment variables and create a connection
load_dotenv()
supabase = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_KEY'])
db = supabase.schema('alp')

In [36]:
# Function to converts dataframes to JSON payload to send to supabase
def to_records(df):
    df = df.copy()
    for c in df.select_dtypes('datetime').columns:
        df[c] = df[c].dt.strftime('%Y-%m-%d')
    return df.astype(object).where(df.notna(), None).to_dict('records')

In [37]:
# Insert JSON records into supabase tables
db.table('hoas_export').upsert(to_records(df_hoas)).execute()
db.table('vendors_intake').upsert(to_records(df_vendors)).execute()
db.table('board_members').delete().gt('id', 0).execute()
db.table('board_members').insert(to_records(df_board)).execute()
db.table('vendor_hoas_junction').upsert(to_records(df_junction)).execute()


APIResponse(data=[{'email': 'contact@brightpathscapes.example', 'hoa_code': 'SR-01'}, {'email': 'contact@brightpathscapes.example', 'hoa_code': 'SR-04'}, {'email': 'contact@brightpathscapes.example', 'hoa_code': 'SR-07'}, {'email': 'contact@brightpathscapes.example', 'hoa_code': 'SR-10'}, {'email': 'dispatch@coastlineplumb.example', 'hoa_code': 'SR-04'}, {'email': 'dispatch@coastlineplumb.example', 'hoa_code': 'SR-07'}, {'email': 'dispatch@coastlineplumb.example', 'hoa_code': 'SR-10'}, {'email': 'service@aquapro.example', 'hoa_code': 'SR-10'}, {'email': 'service@aquapro.example', 'hoa_code': 'SR-12'}, {'email': 'office@reliableelec.example', 'hoa_code': 'SR-01'}, {'email': 'estimates@titanpaving.example', 'hoa_code': 'SR-04'}, {'email': 'dispatch@anchorsecurity.example', 'hoa_code': 'SR-04'}, {'email': 'dispatch@anchorsecurity.example', 'hoa_code': 'SR-10'}], count=None)